In [1]:
import pandas as pd

raw = pd.read_csv('Artworks.csv')

raw.head()

,Title,Artist,ConstituentID,ArtistBio,Nationality,BeginDate,EndDate,Gender,Date,Medium,...,OnView,Circumference (cm),Depth (cm),Diameter (cm),Height (cm),Length (cm),Weight (kg),Width (cm),Seat Height (cm),Duration (sec.)
0,"Ferdinandsbrücke Project, Vienna, Austria (Ele...",Otto Wagner,6210,"(Austrian, 1841–1918)",(Austrian),(1841),(1918),(male),1896,Ink and cut-and-pasted painted pages on paper,...,NaN,NaN,NaN,NaN,48.6000,NaN,NaN,168.9000,NaN,NaN
1,"City of Music, National Superior Conservatory ...",Christian de Portzamparc,7470,"(French, born 1944)",(French),(1944),(0),(male),1987,Paint and colored pencil on print,...,NaN,NaN,NaN,NaN,40.6401,NaN,NaN,29.8451,NaN,NaN
2,"Villa project, outside Vienna, Austria (Elevat...",Emil Hoppe,7605,"(Austrian, 1876–1957)",(Austrian),(1876),(1957),(male),1903,"Graphite, pen, color pencil, ink, and gouache ...",...,NaN,NaN,NaN,NaN,34.3000,NaN,NaN,31.8000,NaN,NaN
3,"The Manhattan Transcripts Project, New York, N...",Bernard Tschumi,7056,"(French and Swiss, born Switzerland 1944)",(),(1944),(0),(male),1980,Photographic reproduction with colored synthet...,...,NaN,NaN,NaN,NaN,50.8000,NaN,NaN,50.8000,NaN,NaN
4,"Villa project, outside Vienna, Austria (Exteri...",Emil Hoppe,7605,"(Austrian, 1876–1957)",(Austrian),(1876),(1957),(male),1903,"Graphite, color pencil, ink, and gouache on tr...",...,NaN,NaN,NaN,NaN,38.4000,NaN,NaN,19.1000,NaN,NaN


In [33]:
import re

import pandas as pd
import re

def extract_countries(artist_bio):

    if pd.isna(artist_bio):
        return []

    # extract each (...) block separately
    artist_blocks = re.findall(r"\(([A-Z][a-zA-Z\s-]+),", artist_bio) #r"\(([A-Z][a-zA-Z\s-]+),"

    countries = []

    for block in artist_blocks:

        # take text before first comma
        country = block.split(",")[0].strip()

        countries.append(country)

    return list(set(countries))

In [34]:
raw["countries"] = raw["ArtistBio"].apply(extract_countries)

In [35]:
cowork =  raw[raw["countries"].apply(len) >= 2]
print(cowork["countries"].to_string())

110                                      [American, French]
111                                      [American, French]
112                                      [American, French]
113                                      [American, French]
154                                        [Dutch, British]
242                                        [Dutch, British]
320                                        [Dutch, British]
322                                        [Dutch, British]
329                                        [Dutch, British]
332                                        [Dutch, British]
334                                        [Dutch, British]
336                                        [Dutch, British]
338                                        [Dutch, British]
341                                        [Dutch, British]
344                                        [Dutch, British]
347                                        [Dutch, British]
351                                     

In [40]:
#Generate country pairs
from itertools import combinations
edges = []


for countries in cowork["countries"]:

    pairs = combinations(countries, 2)

    for source, target in pairs:

        edges.append([source, target])

edges_df = pd.DataFrame(edges, columns=["Source", "Target"])

edges_df = (
    edges_df
    .value_counts()
    .reset_index(name="Weight")
)

nodes = pd.unique(
    edges_df[["Source", "Target"]].values.ravel()
)
nodes_df = pd.DataFrame({
    "Id": nodes,
    "Label": nodes
})

In [41]:
from datetime import datetime
from datetime import datetime

timestamp = datetime.now().strftime("%Y%m%d_%H%M%S")

edges_df.to_csv(f"edges_{timestamp}.csv", index=False)

nodes_df.to_csv(f"nodes_{timestamp}.csv", index=False)

## method 1 : nationality colunm
- select rows with more than 1 ();
        many nan data

## method 2 : artists bio
- named entity reconition

## method3 : artists.csv
- join on artists name 
- identical names? artists info no more than nationality colunm?

In [ ]:
# Run the full preprocessing pipeline (writes JSON for the globe app)
import subprocess
import sys
from pathlib import Path

script = Path("preprocessing/build_network.py")
result = subprocess.run([sys.executable, str(script)], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print(result.stderr)
else:
    print("Done — open frontend/ and run: npm run dev")